# Part 1: Corner Point Method (70 points)

This part is adobpted from [optimization assignment](https://www.cs.cmu.edu/~15381-s19/assignments/programming/optimization/index.html) of AI course at Carnegie Mellon University.

Consider the following problem:


$ Maximize \ x+y $

sobject to:

$ 2x_1 + x_2  >= 0 \\
3x_1 - x_2  <= 14 \\
4x_1 - 2x_2 >= 6 \\
x_1  + 2x_2 <= 15 \\
6x_1 + 3x_2 <= 45 \\
x_1       <= 5 $

\\


---


we can rewrite the problem as:

$ -2x_1 - x_2  <= 0 \\
3x_1 - x_2  <= 14 \\
-4x_1 + 2x_2 <= -6 \\
x_1  + 2x_2 <= 15 \\
6x_1 + 3x_2 <= 45 \\
x_1       <= 5 $

\\

---

\begin{align} A = \begin{pmatrix}
-2 & -1 \\
3 & -1 \\
-4 & 2 \\
1 & 2\\
6 & 3\\
1 & 0 \\
\end{pmatrix}     \end{align}


\begin{align} b = \begin{pmatrix}
0 \\
14 \\
-6 \\
15\\
45\\
5 \\
\end{pmatrix}     \end{align}

\begin{align} c = \begin{pmatrix}
1 \\
1 \\
\end{pmatrix}     \end{align}

\begin{align} x = \begin{pmatrix}
x_1 \\
x_2 \\
\end{pmatrix}     \end{align}


\begin{align} \begin{pmatrix}
-2 & -1 \\
3 & -1 \\
-4 & 2 \\
1 & 2\\
6 & 3\\
1 & 0 \\
\end{pmatrix}   \begin{pmatrix}
x_1 \\
x_2 \\
\end{pmatrix} <= \begin{pmatrix}
0 \\
14 \\
-6 \\
15\\
45\\
5 \\
\end{pmatrix}    \end{align}

You can see the polyhidron and feasible area here :


[visualization](https://quickmath.com/webMathematica3/quickmath/graphs/inequalities/basic.jsp#c=solve&v1=-2x-y%255Cle0%255Cnl%25203x-y%255Cle14%255Cnl%2520-4x%2B2y%255Cle-6%255Cnl%2520x%2B2y%255Cle15%255Cnl%25206x%2B3y%255Cle45%255Cnl%2520x%255Cle5%255Cnl%2520)

You will probably want to take advantage of numpy library for doing linear algebra in Python. The following numpy operations will be particularly useful:

*   **np.dot(A,x):** Matrix-vector multiplication \\
*   **np.linalg.solve(A,b):** Solve the system of linear equations $Ax=b \ for \ x$ \\
*   **np.linalg.matrix_rank(A):** A matrix has to be square (N×N) and full rank (rank = N) in order for **np.linalg.solve** to work. Note: What is the relationship between rank and intersecting hyperplanes (defined by rows of A and b)? \\
*   **A[(1,3,5), :]:** Select the first, third, and fifth rows of 
A (and all columns)

# Part 1: Corner Point Method

In [39]:
import numpy as np

A = np.array([[-2, -1], [3, -1], [-4, 2], [1, 2], [6, 3], [1, 0]])
b = np.array([[0, 14, -6, 15, 45, 5]]).T
C = np.array([[1, 1]]).T

# select 2 rows from 6 of them
import itertools

all_combs = list(itertools.combinations([0, 1, 2, 3, 4, 5], 2))

In [ ]:
# Grade: 50 points
feasible_vertices = []
objectives = []
for comb in all_combs:
    # 1. select a a subset of rows and all columns
    A_sub = A[list(comb), :]
    b_sub = b[list(comb)].squeeze()
    # 2. check if matrix is full rank
    if np.linalg.matrix_rank(A_sub) < A_sub.shape[1]:
        # not full rank => no unique intersection of these two constraints
        continue
    # 3. if the matrix is full rank it certainly has a unique solution. find a solution.
    x = np.linalg.solve(A_sub, b_sub)
    # 4. check if the solution is feasible
    # Tip: Because it is problematic to compare floating point values, you will still want to allow values to be feasible if they are within 1e-6 of their corresponding limit.
    # Tip: You might want to use np.all() to compare two vectors
    if np.all(np.dot(A, x) <= b.squeeze() + 1e-6):
        # 5. if the solution was feasible append it to the list of feasible vertices
        feasible_vertices.append(x)
        # 6. if the solution was feasible append the value of objective to the list of objectives
        obj_val = float(np.dot(C.T, x).item())
        objectives.append(obj_val)

In [44]:
# Grade: 10 points
# find the vertex with maximum objective function
if feasible_vertices:
    best_idx = int(np.argmax(objectives))
    x_best = feasible_vertices[best_idx]
    obj_best = objectives[best_idx]
    print("Optimal vertex:")
    print("x1 =", x_best[0])
    print("x2 =", x_best[1])
    print("objective (x1 + x2) =", obj_best)
else:
    print("No feasible vertices found.")

Optimal vertex:
x1 = 5.0
x2 = 5.0
objective (x1 + x2) = 10.0


Grade: 10 points
### 1. How many constraints are active at the solution?

At the solution (5.0, 5.0), three constraints are active. These are:

- Constraint 4: $x_1 + 2x_2 \leq 15$ → $5 + 2(5) = 15$; equality holds.
- Constraint 5: $6x_1 + 3x_2 \leq 45$ → $6(5) + 3(5) = 45$; equality holds.
- Constraint 6: $x_1 \leq 5$ → $5 = 5$; equality holds.

### 2. which combination of constraints did NOT provide a vertex ? explain why that happened. u can use the visualization to check your answer

The combination of constraints (0, 4) does not provide a vertex because these constraints are parallel, or linearly dependent, meaning their corresponding lines do not intersect at a unique point. Specifically:

- Constraint 0: $-2x_1 - x_2 \leq 0$
- Constraint 4: $6x_1 + 3x_2 \leq 45$

The normal vectors of these constraints, $[-2, -1]$ and $[6, 3]$, are scalar multiples (since $ [6, 3] = -3 \times [-2, -1]$). Therefore, the matrix $A_{\text{sub}}$ for this combination has a rank less than 2, and `np.linalg.solve` cannot find a unique solution. In a visualization, these lines are parallel and never intersect, so they cannot form a vertex.

### 3. which combination of constraints did NOT provide a feasible answer (name just one if there are many) explain why that happend. you can use the visualization to check your answer.

The combination of constraints (2, 5) did not provide a feasible answer. This combination passed the rank check (so it has a unique intersection point), but the point is infeasible because it violates other constraints. Specifically:
- Constraint 2: $-4x_1 + 2x_2 \leq -6$
- Constraint 5: $x_1 \leq 5$

Solving this system gives the point $(5, 7)$. However, this point violates constraint 4 ($x_1 + 2x_2 \leq 15$) because $5 + 2(7) = 19 > 15$. In a visualization, the point $(5, 7)$ lies outside the feasible region, as it is above the line defined by constraint 4. Thus, it is not a feasible vertex.

# Part 2: Online Advertising Optimization (50 points, 30 bonus/optional points)

This part is adobpted from [optimization lectures](https://ocw.mit.edu/courses/15-071-the-analytics-edge-spring-2017/pages/linear-optimization/) of The Analytics Edge course at MIT university.


### Background

When you search on Google, the ads you see next to your search results are chosen by an optimization system.
Each **advertiser** bids different amounts for clicks on different **queries** (keywords), depending on how valuable those clicks are.
Google must decide **how many times to show each advertiser’s ad for each query** to **maximize the estimated total revenue**, while ensuring that:

* each advertiser stays within their **total budget**, and
* Google **cannot display more ads for a query than there are searches** for that query.

For example, if Google estimates that “laptop deals” will be searched about 700 times in a day, it would make no sense to plan for 900 ad impressions on that query — there simply aren’t that many searches available.

---

### Problem Description

You are asked to help Google allocate ad impressions for **3 search queries** and **3 advertisers**:

**Queries:**

1. “laptop deals”
2. “phone accessories”
3. “wireless headphones”

**Advertisers:**

1. **TechWorld** – an electronics retailer
2. **GadgetPro** – sells accessories and peripherals
3. **SoundMax** – specializes in audio devices

---

### Given Data

#### Estimated search volume (available impressions per query)

This represents Google’s estimate of the number of users who will search for that keyword in the given time period. It is the **maximum number of ad impressions** that can be shown for that query.

| Query               | Estimated available impressions |
| :------------------ | :-----------------------------: |
| laptop deals        |               1000              |
| phone accessories   |               800               |
| wireless headphones |               600               |

#### Click-through rates (CTR) — expected clicks per impression

| Query (\backslash) Advertiser | TechWorld | GadgetPro | SoundMax |
| :---------------------------- | :-------: | :-------: | :------: |
| laptop deals                  |    0.05   |    0.02   |   0.03   |
| phone accessories             |    0.04   |    0.06   |   0.01   |
| wireless headphones           |    0.02   |    0.01   |   0.07   |

#### Bids (value per click) and budgets

| Query (\backslash) Advertiser | TechWorld Bid | GadgetPro Bid | SoundMax Bid |
| :---------------------------- | :-----------: | :-----------: | :----------: |
| laptop deals                  |      1.50     |      1.00     |     0.80     |
| phone accessories             |      1.00     |      2.00     |     0.50     |
| wireless headphones           |      0.80     |      0.70     |     1.50     |

| Advertiser | Total Budget ($) |
| :--------- | :--------------: |
| TechWorld  |        300       |
| GadgetPro  |        200       |
| SoundMax   |        150       |

---

### Assignment Tasks & Marks

1. **Formulate the problem as a linear or integer program** (LP/IP) using the data above.

   * Decide the variables, objective, and constraints in a formal optimization model.
   * **Marks:** 30%

3. **(Optional/Bonus)** Use your solver to **solve the Google ad allocation problem** given above.

   * Report the optimal allocation and total estimated revenue.
   * **Bonus marks:** 30%

# Part 2: Online Advertising Optimization

### 1. Formulate the problem as a linear or integer program (LP/IP) using the data above.

**Objective Function:**  
Maximize the total expected revenue, which is the sum of expected revenue per impression (click-through rate multiplied by bid) across all advertiser-query pairs. The objective function is:

$
\text{Maximize } Z = \sum_{q,a} (\text{CTR}_{q,a} \times \text{Bid}_{q,a}) \times x_{q,a}
$

or equivalently,

$
\text{Maximize } Z = 0.075x_{TL} + 0.04x_{TP} + 0.016x_{TH} + 0.02x_{GL} + 0.12x_{GP} + 0.007x_{GH} + 0.024x_{SL} + 0.005x_{SP} + 0.105x_{SH}
$

**Constraints:**  
1. **Budget Constraints** (ensure expected cost per advertiser does not exceed budget):  
   - TechWorld: $ 0.075x_{TL} + 0.04x_{TP} + 0.016x_{TH} \leq 300 $  
   - GadgetPro: $ 0.02x_{GL} + 0.12x_{GP} + 0.007x_{GH} \leq 200 $  
   - SoundMax: $ 0.024x_{SL} + 0.005x_{SP} + 0.105x_{SH} \leq 150 $  

2. **Impression Constraints** (ensure total impressions per query do not exceed search volume):  
   - Laptop deals: $ x_{TL} + x_{GL} + x_{SL} \leq 1000 $  
   - Phone accessories: $ x_{TP} + x_{GP} + x_{SP} \leq 800 $  
   - Wireless headphones: $ x_{TH} + x_{GH} + x_{SH} \leq 600 $  

3. **Non-Negativity Constraints** (impressions cannot be negative):  
   - $ x_{TL} \geq 0 $, $ x_{TP} \geq 0 $, $ x_{TH} \geq 0 $  
   - $ x_{GL} \geq 0 $, $ x_{GP} \geq 0 $, $ x_{GH} \geq 0 $  
   - $ x_{SL} \geq 0 $, $ x_{SP} \geq 0 $, $ x_{SH} \geq 0 $  